# Motional dephasing of a trapped-ion mode (Lindblad)

A damped and dephased quantum harmonic oscillator (a trapped-ion motional mode coupled to a
thermal bath), following the appendix of a trapped-ion coherence paper. No coherent Hamiltonian
here -- this is pure decoherence, as in a Ramsey interrogation:

$$ \dot\rho = \dot\rho_a + \dot\rho_p $$
$$ \dot\rho_a = \frac{\gamma_a \bar n}{2}(2a^\dagger\rho a - \{aa^\dagger,\rho\}) + \frac{\gamma_a(\bar n+1)}{2}(2a\rho a^\dagger - \{a^\dagger a,\rho\}) $$
$$ \dot\rho_p = \frac{\gamma_p}{2}(2 a^\dagger a\, \rho\, a^\dagger a - \{(a^\dagger a)^2,\rho\}) $$

$\bar n$ is the thermal occupation, $\dot{\bar n} = \gamma_a\bar n$ is the heating rate,
$\gamma_p$ the pure dephasing rate. The exact leading-order decay rate of the coherence between
Fock states $|0\rangle,|1\rangle$ follows from the three dissipators directly
(cooling contributes $\gamma_a(\bar n+1)/2$, heating $\tfrac{3}{2}\gamma_a\bar n$, dephasing $\gamma_p/2$):

$$ \frac{1}{T_2^{\text{lead}}} = 2\dot{\bar n} + \frac{\gamma_a}{2} + \frac{\gamma_p}{2} $$

This is a true **lower bound** on the numerically exact $T_2$: the only correction is heating
re-feeding coherence from higher Fock-state pairs ($\rho_{12} \to \rho_{01}$), which can only
*slow* the decay. The often-quoted approximation $T_2 \approx 1/(2\dot{\bar n} + \gamma_p/2)$
drops the spontaneous-decay piece $\gamma_a/2$ and is therefore **not** a bound in general --
at $\bar n \lesssim 1$ the exact $T_2$ comes out *below* it. It's only valid for
$\bar n \gg 1$, where $\gamma_a \ll \dot{\bar n}$.

## Design: htdse's Lindblad support

`Mechanism.jump_operators(t)` defaults to `[]` (closed system); a mechanism modeling
dissipation into an unmodeled/uncharacterized bath overrides it, returning each $L_k$
already scaled by $\sqrt{\text{rate}}$. `LindbladEvolution(mechanism, rho0)` then solves
$\dot\rho = -i[H,\rho] + \sum_k L_k\rho L_k^\dagger - \tfrac12\{L_k^\dagger L_k,\rho\}$ with the
same lazy, extend-on-demand, never-extrapolate, verbose-by-default solver used everywhere else
in `core/` -- it shares its internal engine with `HamiltonianEvolution`/`UnitaryEvolution`,
just with a Lindblad right-hand side instead of a Schrodinger one.

`submodules/harmonic_oscillator.py`'s `ThermalMotionalDecoherence` implements exactly the three
jump operators above: $L_1=\sqrt{\gamma_a\bar n}\,a^\dagger$ (heating),
$L_2=\sqrt{\gamma_a(\bar n+1)}\,a$ (cooling), $L_3=\sqrt{\gamma_p}\,a^\dagger a$ (dephasing).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from htdse.core.operator import Operator
from htdse.core.evolution import LindbladEvolution
from htdse.submodules.harmonic_oscillator import fock, ThermalMotionalDecoherence

n_max = 20
psi0 = (fock(0, n_max) + fock(1, n_max)) / np.sqrt(2)
rho0 = Operator(np.outer(psi0, psi0.conj()))

def fit_T2(ts, coherence):
    mask = coherence > 1e-6
    slope, _ = np.polyfit(ts[mask], np.log(coherence[mask]), 1)
    return 1 / (-slope)

## LF mode: heating-dominated ($\dot{\bar n} \approx 50$ quanta/s, no dephasing)

With $\bar n = 1$, $\gamma_a = 50$: lower bound $T_2 \geq 1/(2\times 50 + 25) = 8\,\mathrm{ms}$
(the dropped-$\gamma_a/2$ form would say $10\,\mathrm{ms}$). Expect the numerically exact $T_2$
to come out well *above* the bound -- at $\bar n \sim 1$ heating re-feeds coherence strongly,
so the leading-order rate is least tight here.

In [ ]:
nbar_lf, ndot_lf = 1.0, 50.0
mech_lf = ThermalMotionalDecoherence(n_max, gamma_a=ndot_lf / nbar_lf, nbar=nbar_lf, gamma_p=0.0)
ev_lf = LindbladEvolution(mech_lf, rho0, verbose=False)

ts_lf = np.linspace(0, 0.03, 30)  # seconds
rhos_lf = ev_lf.state_at(ts_lf)
coh_lf = np.abs(rhos_lf[:, 0, 1])
trace_lf = np.real(np.trace(rhos_lf, axis1=1, axis2=2))

print(f"trace stays in [{trace_lf.min():.10f}, {trace_lf.max():.10f}]")
T2_fit_lf = fit_T2(ts_lf, coh_lf)
T2_theory_lf = 1 / (2 * ndot_lf + (ndot_lf / nbar_lf) / 2)  # 2*ndot + gamma_a/2
print(f"fitted T2 = {T2_fit_lf*1e3:.2f} ms, exact-rate lower bound = {T2_theory_lf*1e3:.2f} ms")

plt.semilogy(ts_lf * 1e3, coh_lf, "o-")
plt.xlabel("t (ms)")
plt.ylabel(r"$|\rho_{01}(t)|$")
plt.title("LF mode: heating-dominated coherence decay")
plt.show()

## HF mode: dephasing-dominated ($\dot{\bar n} \approx 1.6$ quanta/s, $\gamma_p \approx 2\pi\times 2.3$ Hz)

Lower bound: $T_2 \geq 1/(2\times 1.6 + 0.08 + \pi\times 2.3) \approx 95.2\,\mathrm{ms}$
($\gamma_a/2 = 0.08$ is negligible here -- this is the $\bar n \gg 1$ regime where the
short form is fine). Heating is slow, so the bound should be nearly saturated.

In [ ]:
nbar_hf, ndot_hf, gamma_p_hf = 10.0, 1.6, 2 * np.pi * 2.3
mech_hf = ThermalMotionalDecoherence(n_max, gamma_a=ndot_hf / nbar_hf, nbar=nbar_hf, gamma_p=gamma_p_hf)
ev_hf = LindbladEvolution(mech_hf, rho0, verbose=False)

ts_hf = np.linspace(0, 0.25, 40)
rhos_hf = ev_hf.state_at(ts_hf)
coh_hf = np.abs(rhos_hf[:, 0, 1])
trace_hf = np.real(np.trace(rhos_hf, axis1=1, axis2=2))

print(f"trace stays in [{trace_hf.min():.10f}, {trace_hf.max():.10f}]")
T2_fit_hf = fit_T2(ts_hf, coh_hf)
T2_theory_hf = 1 / (2 * ndot_hf + (ndot_hf / nbar_hf) / 2 + gamma_p_hf / 2)
print(f"fitted T2 = {T2_fit_hf*1e3:.2f} ms, exact-rate lower bound = {T2_theory_hf*1e3:.2f} ms")

plt.semilogy(ts_hf * 1e3, coh_hf, "o-", color="C1")
plt.xlabel("t (ms)")
plt.ylabel(r"$|\rho_{01}(t)|$")
plt.title("HF mode: dephasing-dominated coherence decay")
plt.show()